<a href="https://colab.research.google.com/github/darlim9141/kcu5/blob/main/backend_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [Study Note] AI Model Serving and Backend Architecture

## 1. Introduction: From Model to Service

### 1.1. Objective
In the previous stages, we successfully trained a ResNet-50 model and saved it as a `.keras` file. However, a model file sitting on a hard drive provides no value to end-users.
The objective of this phase is **Model Serving**: creating a system that accepts user input (images), processes it through the model, and returns the prediction (fashion style) via the web.

### 1.2. Architecture Overview
We adopted a **Microservices Architecture** where the frontend and backend are decoupled.
* **Frontend (Client):** React.js (User Interface)
* **Backend (Server):** FastAPI (Model Inference Engine)
* **Communication:** RESTful API (HTTP Request/Response)

## 2. Web Framework: FastAPI

### 2.1. Why FastAPI?
We selected **FastAPI** over traditional frameworks like Flask or Django for the following reasons:

1.  **Asynchronous I/O (Async/Await):** Machine Learning inference is often a blocking operation. FastAPI supports asynchronous programming, allowing the server to handle other requests while waiting for the model or database, significantly improving **Concurrency**.
2.  **Performance:** Built on **Starlette** (ASGI) and **Pydantic**, it offers performance on par with NodeJS and Go.
3.  **Automatic Documentation:** It automatically generates interactive API documentation (Swagger UI), which facilitates easy testing and collaboration with frontend developers.

### 2.2. Implementation Logic
The backend server follows this logic flow:
1.  **Startup:** Load the heavy AI model into memory (RAM) *once* when the server starts or upon the first request (Lazy Loading) to avoid overhead on every call.
2.  **Endpoint (`/analyze`):** Receive an image file via POST request.
3.  **Preprocessing:** Convert the raw bytes into an image tensor and normalize it (same preprocessing as training).
4.  **Inference:** Pass the tensor to the model to get probability scores.
5.  **Response:** Return the class with the highest probability as a JSON object.

In [ ]:
# Conceptual implementation of main.py
# Note: This code serves as an architectural example.

from fastapi import FastAPI, UploadFile, File
import uvicorn
import numpy as np
from PIL import Image
import io

app = FastAPI()

# 1. Model Loading (Singleton Pattern)
# We load the model globally so it persists across requests.
model = None

def load_model():
    global model
    # model = tf.keras.models.load_model('model.keras')
    print("Model loaded into memory.")

@app.on_event("startup")
async def startup_event():
    load_model()

# 2. API Endpoint Definition
@app.post("/api/analyze")
async def analyze_image(file: UploadFile = File(...)):
    """
    Receives an image, performs inference, and returns the style.
    """
    # A. Read Image
    image_bytes = await file.read()
    image = Image.open(io.BytesIO(image_bytes))

    # B. Preprocessing (Resize & Normalize)
    image = image.resize((224, 224))
    img_array = np.array(image) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    # C. Inference
    # prediction = model.predict(img_array)
    # style = class_names[np.argmax(prediction)]

    # D. Response
    return {
        "status": "success",
        "style": "Minimal", # Example result
        "confidence": 0.98
    }

## 3. Containerization: Docker

### 3.1. The "Dependency Hell" Problem
Machine Learning projects often depend on specific versions of libraries (e.g., `tensorflow==2.10`, `python==3.9`). Running the code on a different server often leads to compatibility errors.

### 3.2. Solution: Docker
**Docker** packages the application with all its dependencies (Operating System libs, Python runtime, environment variables) into a standardized unit called a **Container**.

* **Dockerfile:** A script containing a series of instructions to build the Docker image. It ensures that the environment is **reproducible** anywhere.

### 3.3. Our Dockerfile Configuration
We used a multi-layered build process to optimize the server environment.

In [ ]:
# --- Dockerfile Content Analysis ---

# 1. Base Image: Use Python 3.10 installed on a lightweight Linux (Debian/Alpine)
FROM python:3.10

# 2. Working Directory: Create a folder inside the container
WORKDIR /code

# 3. System Dependencies: Install OS-level libraries required for OpenCV/Image processing
RUN apt-get update && apt-get install -y libgl1 libglib2.0-0

# 4. Dependency Installation: Copy requirements first to leverage Docker Cache
COPY requirements.txt /code/requirements.txt
RUN pip install --no-cache-dir --upgrade -r /code/requirements.txt

# 5. Copy Source Code: Move actual application code into the container
COPY . /code

# 6. User Permissions: Create a non-root user for security (Requirement for Hugging Face)
RUN useradd -m -u 1000 user
USER user
ENV HOME=/home/user \
    PATH=/home/user/.local/bin:$PATH
WORKDIR $HOME/app
COPY --chown=user . $HOME/app

# 7. Entrypoint: Command to start the uvicorn server
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "7860"]

## 4. Deployment Strategy & Persistence

### 4.1. Managing Large Files (Git LFS)
Standard GitHub repositories block files larger than 100MB. Our trained model (`model.keras`) is approximately 150MB.
* **Solution:** We implemented **Git LFS (Large File Storage)**.
* **Mechanism:** Git tracks a lightweight pointer (text reference) in the repository, while the actual binary file is stored on a separate LFS server.

### 4.2. Cloud Platform: Hugging Face Spaces
We deployed our Docker container to **Hugging Face Spaces**.
* **Benefit:** It provides a free cloud environment capable of running heavy ML workloads (CPU/GPU) and natively supports Docker SDK.

### 4.3. Data Persistence (Stateless vs. Stateful)
Cloud containers are **ephemeral** (stateless); when the server restarts, all local files (e.g., `database.db`) are wiped.
* **Challenge:** We needed to persist user statistics (Total Images, Unique Users) across restarts without a paid database.
* **Implementation:** We utilized **Hugging Face Datasets** as a remote storage solution.
    1.  **Read:** On API request, the backend fetches the `stats.json` from the remote dataset repository.
    2.  **Write:** After updating counts, the backend commits the changes back to the remote repository programmatically using `huggingface_hub`.

This approach ensures data integrity and permanence without incurring additional infrastructure costs.